# 04 — Batch-score every live account and build the Account 360 serving table

* Loads `officepulse_churn.churn_model@champion` from Unity Catalog.
* Scores the **latest week** for every live account (renewal risk + top-3 SHAP drivers).
* Writes `gold_churn_scores` (history, appended per run) and `gold_account_360`
  (one row per account: CRM + contract + latest KPIs + features + score).
* `gold_account_360`, `silver_tickets`, `silver_ticket_events` and `silver_usage_weekly` have
  Change Data Feed enabled and are synced to **Lakebase** (see `05_publish_lakebase.py`) where the
  CS app reads them with millisecond latency.

In [2]:
%pip install -q "mlflow>=2.21,<4" "scikit-learn==1.5.2" "shap==0.46.0" "numpy<2.2"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [3]:
dbutils.library.restartPython()

In [4]:
import json
import mlflow
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from churn_features import FEATURES, build_features, rule_score

dbutils.widgets.text("catalog", "ytcy_azure_east2classic_stable")
dbutils.widgets.text("schema", "officepulse_churn")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
FQ = f"{CATALOG}.{SCHEMA}"
MODEL_URI = f"models:/{FQ}.churn_model@champion"
mlflow.set_registry_uri("databricks-uc")

model = mlflow.pyfunc.load_model(MODEL_URI)
version = mlflow.MlflowClient().get_model_version_by_alias(f"{FQ}.churn_model", "champion").version
print("loaded", MODEL_URI, "version", version)

{"type": "mimeBundle", "data": {"text/plain": "Downloading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]", "application/vnd.jupyter.widget-view+json": {"version_major": 2, "version_minor": 0, "model_id": "c081ebf1a0c7480fb842873b84e3d111"}}, "executionCount": null, "arguments": {}, "addedWidgets": {}, "removedWidgets": [], "datasetInfos": [], "metadata": {"kernelSessionId": "43d7201a-7a185ef5b04c7cc5eaebceb7"}}

loaded models:/ytcy_azure_east2classic_stable.officepulse_churn.churn_model@champion version 5


In [5]:
weekly = spark.table(f"{FQ}.gold_account_weekly").toPandas()
contracts = spark.table(f"{FQ}.silver_contracts").toPandas()
accounts = spark.table(f"{FQ}.silver_accounts").toPandas()
latest_week = pd.to_datetime(weekly.week_start).max()

feat = build_features(weekly, contracts, accounts, with_label=False)
live_ids = set(contracts[contracts.end_date.isna()].account_id)
cur = feat[(feat.snapshot_week == latest_week) & feat.account_id.isin(live_ids)]
# accounts with < 12 weeks of usage are still onboarding: their ramp-up looks like a decline and the
# model was not trained on them -> they are handled by the onboarding playbook, not scored
n_onb = int((cur.weeks_observed < 12).sum())
cur = cur[cur.weeks_observed >= 12].reset_index(drop=True)
print("latest week:", latest_week.date(), " live accounts scored:", len(cur), " onboarding (not scored):", n_onb)

pred = model.predict(cur[FEATURES])
cur["churn_probability"] = pred["churn_probability"].values
cur["top_drivers"] = pred["top_drivers"].values
cur["rule_score"] = rule_score(cur).round(1)
cur["risk_score"] = (cur.churn_probability * 100).round(1)
cur["risk_band"] = np.select([cur.churn_probability >= 0.5, cur.churn_probability >= 0.25], ["高", "中"], "低")

status = accounts.set_index("account_id").status
cur["csm_flagged"] = cur.account_id.map(status).eq("churn_risk")
cur["hidden_risk"] = (cur.risk_band == "高") & ~cur.csm_flagged
cur["model_version"] = int(version)
cur["scored_at"] = pd.Timestamp.utcnow().tz_localize(None)

display(cur.sort_values("churn_probability", ascending=False)
        [["account_id", "churn_probability", "risk_band", "rule_score", "csm_flagged", "hidden_risk", "days_to_renewal", "top_drivers"]].head(25))

latest week: 2026-11-02  live accounts scored: 448  onboarding (not scored): 25


| account_id | churn_probability | risk_band | rule_score | csm_flagged | hidden_risk | days_to_renewal | top_drivers |
|---|---|---|---|---|---|---|---|
| ACC-1259 | 0.9625906679084834 | 高 | 24.7 | False | True | 14 | [{"feature": "attendance_detect_rate_4w", "label": "出社検知率(直近4週)", "value": 0.5785750000000001, "impact": 1.5199}, {"feat |
| ACC-1008 | 0.938054351824144 | 高 | 62.3 | True | False | 28 | [{"feature": "attendance_detect_rate_4w", "label": "出社検知率(直近4週)", "value": 0.45754999999999996, "impact": 1.885}, {"feat |
| ACC-1420 | 0.9315403210555528 | 高 | 44.6 | True | False | 15 | [{"feature": "attendance_detect_rate_4w", "label": "出社検知率(直近4週)", "value": 0.514175, "impact": 2.1379}, {"feature": "avg |
| ACC-1013 | 0.8898963346872265 | 高 | 45.2 | True | False | 56 | [{"feature": "attendance_detect_rate_4w", "label": "出社検知率(直近4週)", "value": 0.58335, "impact": 1.3828}, {"feature": "avg_ |
| ACC-1169 | 0.8471285028563523 | 高 | 14.3 | False | True | 242 | [{"feature": "attendance_dete

In [6]:
score_cols = ["account_id", "snapshot_week", "churn_probability", "risk_score", "risk_band", "rule_score",
              "top_drivers", "csm_flagged", "hidden_risk", "model_version", "scored_at"]
scores_sdf = spark.createDataFrame(cur[score_cols]).withColumn("snapshot_week", F.to_date("snapshot_week"))
scores_sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.gold_churn_scores")
spark.sql(f"COMMENT ON TABLE {FQ}.gold_churn_scores IS 'Renewal-risk scores per account per scoring run (model officepulse_churn.churn_model@champion). top_drivers = JSON array of SHAP drivers.'")

{"type": "mimeBundle", "data": {"text/plain": "DataFrame[]"}, "executionCount": 3, "arguments": {}, "addedWidgets": {}, "removedWidgets": [], "datasetInfos": [], "metadata": {"kernelSessionId": "43d7201a-7a185ef5b04c7cc5eaebceb7"}}

 ## Account 360 (one row per account) — the table the app serves from Lakebase

In [8]:
scores_sdf.createOrReplaceTempView("latest_scores")
spark.createDataFrame(cur[["account_id"] + FEATURES]).createOrReplaceTempView("latest_features")
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.gold_account_360
COMMENT 'Account 360 for the CS app: CRM + contract + latest usage KPIs + ML renewal-risk score and drivers. Synced to Lakebase.'
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
WITH last_usage AS (
  SELECT * FROM {FQ}.silver_usage_weekly
  QUALIFY row_number() OVER (PARTITION BY account_id ORDER BY week_start DESC) = 1
), tk AS (
  SELECT account_id,
         count(*) AS tickets_total,
         sum(CASE WHEN status IN ('new','open','pending') THEN 1 ELSE 0 END) AS open_tickets_now,
         round(avg(csat), 2) AS avg_csat,
         max(is_cancellation_intent) AS has_cancellation_ticket
  FROM {FQ}.silver_tickets GROUP BY account_id
)
SELECT
  a.account_id, a.company_name, a.industry, a.plan, a.status, a.csm_owner, a.seats, a.arr_jpy, a.region,
  c.contract_id, c.start_date, c.end_date, c.renewal_date, c.loaned_devices, c.used_services,
  c.churn_reason, c.mrr_jpy, c.auto_renew,
  u.week_start AS last_usage_week, u.login_rate, u.signup_rate, u.attendance_detect_rate, u.install_rate,
  u.app_config_ok_rate,
  coalesce(tk.tickets_total, 0) AS tickets_total, coalesce(tk.open_tickets_now, 0) AS open_tickets_now,
  tk.avg_csat, coalesce(tk.has_cancellation_ticket, false) AS has_cancellation_ticket,
  s.churn_probability, s.risk_score, s.risk_band, s.rule_score, s.top_drivers, s.csm_flagged, s.hidden_risk,
  s.model_version, s.scored_at,
  CASE WHEN s.churn_probability IS NOT NULL THEN round(s.churn_probability * a.arr_jpy) END AS arr_at_risk_jpy,
  to_json(named_struct({", ".join(f"'{f}', f.{f}" for f in FEATURES)})) AS features_json
FROM {FQ}.silver_accounts a
LEFT JOIN {FQ}.silver_contracts c ON c.account_id = a.account_id
LEFT JOIN last_usage u ON u.account_id = a.account_id
LEFT JOIN tk ON tk.account_id = a.account_id
LEFT JOIN latest_scores s ON s.account_id = a.account_id
LEFT JOIN latest_features f ON f.account_id = a.account_id
""")
spark.sql(f"ALTER TABLE {FQ}.gold_account_360 SET TAGS ('cs_domain' = 'customer_success', 'medallion_layer' = 'gold', 'serving_layer' = 'lakebase')")

# Source tables synced to Lakebase need Change Data Feed for TRIGGERED sync
for t in ["silver_tickets", "silver_ticket_events", "silver_usage_weekly"]:
    try:
        spark.sql(f"ALTER MATERIALIZED VIEW {FQ}.{t} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    except Exception as e:
        print(t, "CDF via ALTER not supported on MV (synced in SNAPSHOT mode):", str(e)[:120])

silver_tickets CDF via ALTER not supported on MV (synced in SNAPSHOT mode): (com.databricks.pipelines.common.errors.DLTSparkException) [SET_TBLPROPERTIES_NOT_ALLOWED_FOR_PIPELINE_TABLE] ALTER MATE
silver_ticket_events CDF via ALTER not supported on MV (synced in SNAPSHOT mode): (com.databricks.pipelines.common.errors.DLTSparkException) [SET_TBLPROPERTIES_NOT_ALLOWED_FOR_PIPELINE_TABLE] ALTER MATE
silver_usage_weekly CDF via ALTER not supported on MV (synced in SNAPSHOT mode): (com.databricks.pipelines.common.errors.DLTSparkException) [SET_TBLPROPERTIES_NOT_ALLOWED_FOR_PIPELINE_TABLE] ALTER MATE


In [9]:
summary = spark.sql(f"""
SELECT risk_band, count(*) AS accounts, sum(arr_jpy) AS arr_jpy, round(sum(arr_at_risk_jpy)) AS arr_at_risk_jpy,
       sum(CASE WHEN csm_flagged THEN 1 ELSE 0 END) AS csm_flagged, sum(CASE WHEN hidden_risk THEN 1 ELSE 0 END) AS hidden_risk
FROM {FQ}.gold_account_360 WHERE churn_probability IS NOT NULL GROUP BY risk_band ORDER BY risk_band
""")
display(summary)
out = {r["risk_band"]: {k: (float(v) if v is not None else None) for k, v in r.asDict().items() if k != "risk_band"} for r in summary.collect()}

| risk_band | accounts | arr_jpy | arr_at_risk_jpy | csm_flagged | hidden_risk |
|---|---|---|---|---|---|
| 中 | 62 | 234910000 | 81046591.0 | 0 | 0 |
| 低 | 345 | 1751050000 | 119433567.0 | 10 | 0 |
| 高 | 41 | 119810000 | 79558257.0 | 5 | 36 |


In [10]:
dbutils.notebook.exit(json.dumps({"scored_accounts": len(cur), "latest_week": str(latest_week.date()), "by_band": out}, ensure_ascii=False))

{"scored_accounts": 448, "latest_week": "2026-11-02", "by_band": {"中": {"accounts": 62.0, "arr_jpy": 234910000.0, "arr_at_risk_jpy": 81046591.0, "csm_flagged": 0.0, "hidden_risk": 0.0}, "低": {"accounts": 345.0, "arr_jpy": 1751050000.0, "arr_at_risk_jpy": 119433567.0, "csm_flagged": 10.0, "hidden_risk": 0.0}, "高": {"accounts": 41.0, "arr_jpy": 119810000.0, "arr_at_risk_jpy": 79558257.0, "csm_flagged": 5.0, "hidden_risk": 36.0}}}